# 15 — Pré-processamento ML: Join IDHM + Splits

**Input:** `features_socioeconomicas.parquet` (saída do notebook 13)  
**Output:**
- `ml_features.parquet` — todas as features + IDHM + `_row_id`
- `test_set.parquet` — conjunto de teste fixo (mesmo para todos os classificadores)
- `val_set.parquet` — conjunto de validação fixo

**Definição das variantes de features:**
| Variante | Features | Qtd |
|----------|---------|-----|
| M1 | ENEM individual (Q_*, TP_*, REGIAO) | 28 |
| M2 | M1 + IDHM agregado por UF | 37 |
| M3 | M2 + `idhm_cand_raca` + `idhm_cand_sexo` | 39 |

**IDHM personalizado (M3):**
- `idhm_cand_raca`: candidato branco → idhm_branco; preto/pardo → idhm_negro
- `idhm_cand_sexo`: masculino → idhm_homem; feminino → idhm_mulher

**Split estratificado por ano (2019-2024):** 80% treino / 10% validação / 10% teste

In [1]:
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

BASE         = Path('../data/processed')
FEAT_PARQUET = BASE / 'features_socioeconomicas.parquet'
ATLAS_UF     = BASE / 'atlas/atlas_uf.parquet'
ML_PARQUET   = BASE / 'ml_features.parquet'
TEST_SET     = BASE / 'test_set.parquet'
VAL_SET      = BASE / 'val_set.parquet'
SEED         = 42

print(f'features_socioeconomicas: {FEAT_PARQUET.stat().st_size/1024**2:.0f} MB')
print(f'atlas_uf:                 {ATLAS_UF.stat().st_size/1024**2:.1f} MB')

features_socioeconomicas: 631 MB
atlas_uf:                 0.0 MB


## 1. Verificação de schemas

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit = '6GB'")
con.execute("SET threads = 4")

print('=== atlas_uf.parquet ===')
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ATLAS_UF}')").df()[['column_name','column_type']].to_string())

print('\n=== Anos disponíveis no atlas ===')
print(con.execute(f"SELECT DISTINCT ano FROM read_parquet('{ATLAS_UF}') ORDER BY ano").df().T)

print('\n=== features_socioeconomicas (primeiras colunas) ===')
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FEAT_PARQUET}')").df()[['column_name','column_type']].to_string())

=== atlas_uf.parquet ===
             column_name column_type
0                  sg_uf     VARCHAR
1                    ano      BIGINT
2                   idhm      DOUBLE
3          idhm_educacao      DOUBLE
4             idhm_renda      DOUBLE
5       idhm_longevidade      DOUBLE
6        renda_percapita      DOUBLE
7       tx_analfabetismo      DOUBLE
8      tx_envelhecimento      DOUBLE
9         esperanca_vida      DOUBLE
10  mortalidade_infantil      DOUBLE
11           idhm_branco      DOUBLE
12            idhm_homem      DOUBLE
13           idhm_mulher      DOUBLE
14            idhm_negro      DOUBLE
15            idhm_rural      DOUBLE
16           idhm_urbano      DOUBLE

=== Anos disponíveis no atlas ===
       0     1     2     3     4     5     6     7     8     9     10    11  \
ano  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023   

       12  
ano  2024  

=== features_socioeconomicas (primeiras colunas) ===
             column_name column_type


## 2. Join: features + IDHM → ml_features.parquet

Join por `SG_UF_ESC` (UF da escola) e `NU_ANO`.  
LEFT JOIN para manter todos os candidatos (candidatos sem escola registrada terão IDHM = NULL).

In [3]:
print('Criando ml_features.parquet...')
import time; t0 = time.time()

con.execute(f"""
    COPY (
        SELECT
            row_number() OVER (ORDER BY f.NU_ANO, f.SG_UF_ESC) AS _row_id,
            f.*,
            -- M2: IDHM agregado por UF
            a.idhm,
            a.idhm_educacao,
            a.idhm_renda,
            a.idhm_longevidade,
            a.renda_percapita,
            a.tx_analfabetismo,
            a.tx_envelhecimento,
            a.esperanca_vida,
            a.mortalidade_infantil,
            -- M3: IDHM personalizado por raça/sexo do candidato
            -- TP_COR_RACA_NUM: 1=Branca, 2=Preta, 3=Parda, 4=Amarela, 5=Indígena
            CASE
                WHEN f.TP_COR_RACA_NUM = 1         THEN a.idhm_branco
                WHEN f.TP_COR_RACA_NUM IN (2, 3)   THEN a.idhm_negro
            END AS idhm_cand_raca,
            -- TP_SEXO_NUM: 0=Masculino, 1=Feminino
            CASE
                WHEN f.TP_SEXO_NUM = 0 THEN a.idhm_homem
                WHEN f.TP_SEXO_NUM = 1 THEN a.idhm_mulher
            END AS idhm_cand_sexo
        FROM read_parquet('{FEAT_PARQUET}') f
        LEFT JOIN read_parquet('{ATLAS_UF}') a
            ON f.SG_UF_ESC = a.sg_uf
           AND f.NU_ANO    = a.ano
    )
    TO '{ML_PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f'Concluído em {(time.time()-t0)/60:.1f} min')
print(f'Tamanho: {ML_PARQUET.stat().st_size/1024**2:.0f} MB')

Criando ml_features.parquet...
Concluído em 17.9 min
Tamanho: 651 MB


## 3. Cobertura das novas features

In [4]:
df_cov = con.execute(f"""
    SELECT
        NU_ANO,
        COUNT(*) AS n_total,
        ROUND(AVG(CASE WHEN idhm            IS NOT NULL THEN 100.0 ELSE 0 END), 1) AS pct_idhm,
        ROUND(AVG(CASE WHEN idhm_cand_raca  IS NOT NULL THEN 100.0 ELSE 0 END), 1) AS pct_raca,
        ROUND(AVG(CASE WHEN idhm_cand_sexo  IS NOT NULL THEN 100.0 ELSE 0 END), 1) AS pct_sexo
    FROM read_parquet('{ML_PARQUET}')
    GROUP BY NU_ANO ORDER BY NU_ANO
""").df()

print('Cobertura das features IDHM por ano:')
print(df_cov.to_string(index=False))

print('\nDistribuição FAIXA_MT (2019-2024):')
df_dist = con.execute(f"""
    SELECT FAIXA_MT, COUNT(*) AS n
    FROM read_parquet('{ML_PARQUET}')
    WHERE NU_ANO >= 2019 AND FAIXA_MT IS NOT NULL
    GROUP BY FAIXA_MT ORDER BY FAIXA_MT
""").df()
df_dist['pct'] = (df_dist['n'] / df_dist['n'].sum() * 100).round(1)
print(df_dist.to_string(index=False))

Cobertura das features IDHM por ano:
 NU_ANO  n_total  pct_idhm  pct_raca  pct_sexo
   2012  4079886      30.8      29.4      30.8
   2013  5007934      27.2      26.0      27.2
   2014  5947909      24.5      23.5      24.5
   2015  5604905      25.4      24.3      25.4
   2016  5818264      26.1      24.9      26.1
   2017  4426692      31.3      29.8      31.3
   2018  3893729      29.0      27.6      29.0
   2019  3701910      25.8      24.5      25.8
   2020  2588681      20.8      19.8      20.8
   2021  2238107      26.9      25.8      26.9
   2022  2344823      29.6      28.4      29.6
   2023  2678264      26.9      26.1      26.9
   2024  2990093      39.9      38.6      39.9

Distribuição FAIXA_MT (2019-2024):
FAIXA_MT       n  pct
 400-500 5782869 35.0
 500-600 4213369 25.5
 600-700 3108187 18.8
 700-800 1194763  7.2
    <400 1956070 11.8
    >800  286620  1.7


## 4. Criação dos splits estratificados por ano

Para cada ano (2019-2024): 10% teste, 10% validação, 80% treino.  
Estratificado por `FAIXA_3C` para manter distribuição de classes em cada split.

Os arquivos `test_set.parquet` e `val_set.parquet` são **fixos** — todos os
classificadores usarão exatamente as mesmas linhas.

In [5]:
MAP_3C = {
    '<400': 'Baixo', '400-500': 'Baixo',
    '500-600': 'Medio', '600-700': 'Medio',
    '700-800': 'Alto', '>800': 'Alto',
}

# Carrega apenas _row_id + NU_ANO + FAIXA_MT (leve)
print('Carregando IDs para split...')
df_ids = con.execute(f"""
    SELECT _row_id, NU_ANO, FAIXA_MT
    FROM read_parquet('{ML_PARQUET}')
    WHERE FAIXA_MT IS NOT NULL AND NU_ANO BETWEEN 2019 AND 2024
""").df()
df_ids['faixa_3c'] = df_ids['FAIXA_MT'].map(MAP_3C)
print(f'Total: {len(df_ids):,} linhas')

test_rids, val_rids = [], []
resumo = []

for ano in sorted(df_ids['NU_ANO'].unique()):
    grp = df_ids[df_ids['NU_ANO'] == ano].copy()
    # 90% remainder + 10% test
    remainder, test_part = train_test_split(
        grp, test_size=0.10, stratify=grp['faixa_3c'], random_state=SEED)
    # ~89% train + ~10% val (do remainder)
    _, val_part = train_test_split(
        remainder, test_size=0.10, stratify=remainder['faixa_3c'], random_state=SEED)

    test_rids.extend(test_part['_row_id'].tolist())
    val_rids.extend(val_part['_row_id'].tolist())
    resumo.append({
        'Ano': ano, 'Total': len(grp),
        'Teste': len(test_part), 'Val': len(val_part),
        'Treino pool': len(grp) - len(test_part) - len(val_part),
    })

df_resumo = pd.DataFrame(resumo)
print('\nResumo dos splits por ano:')
print(df_resumo.to_string(index=False))
print(f'\nTotal teste: {len(test_rids):,} | Total val: {len(val_rids):,}')

Carregando IDs para split...
Total: 16,541,878 linhas

Resumo dos splits por ano:
 Ano   Total  Teste    Val  Treino pool
2019 3701910 370191 333172      2998547
2020 2588681 258869 232982      2096830
2021 2238107 223811 201430      1812866
2022 2344823 234483 211034      1899306
2023 2678264 267827 241044      2169393
2024 2990093 299010 269109      2421974

Total teste: 1,654,191 | Total val: 1,488,771


In [6]:
# Salva test_set.parquet e val_set.parquet como linhas completas
test_ids_df = pd.DataFrame({'_row_id': test_rids})
val_ids_df  = pd.DataFrame({'_row_id': val_rids})

con.register('_test_ids', test_ids_df)
con.register('_val_ids',  val_ids_df)

print('Salvando test_set.parquet...')
t0 = time.time()
con.execute(f"""
    COPY (
        SELECT ml.* FROM read_parquet('{ML_PARQUET}') ml
        INNER JOIN _test_ids t USING (_row_id)
    ) TO '{TEST_SET}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f'  {TEST_SET.stat().st_size/1024**2:.0f} MB  ({time.time()-t0:.1f}s)')

print('Salvando val_set.parquet...')
t0 = time.time()
con.execute(f"""
    COPY (
        SELECT ml.* FROM read_parquet('{ML_PARQUET}') ml
        INNER JOIN _val_ids v USING (_row_id)
    ) TO '{VAL_SET}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f'  {VAL_SET.stat().st_size/1024**2:.0f} MB  ({time.time()-t0:.1f}s)')

con.close()
print('\nPronto!')

Salvando test_set.parquet...
  27 MB  (2.2s)
Salvando val_set.parquet...
  24 MB  (2.1s)

Pronto!


## 5. Verificação dos splits

In [7]:
import pandas as pd

df_test_chk = pd.read_parquet(TEST_SET, columns=['NU_ANO', 'FAIXA_MT'])
df_val_chk  = pd.read_parquet(VAL_SET,  columns=['NU_ANO', 'FAIXA_MT'])

print(f'test_set:  {len(df_test_chk):,} linhas | anos: {sorted(df_test_chk.NU_ANO.unique())}')
print(f'val_set:   {len(df_val_chk):,} linhas  | anos: {sorted(df_val_chk.NU_ANO.unique())}')

print('\nDistribuição FAIXA_MT no teste:')
df_test_chk['faixa_3c'] = df_test_chk['FAIXA_MT'].map(MAP_3C)
print(df_test_chk['faixa_3c'].value_counts(normalize=True).apply(lambda x: f'{x:.1%}'))

# Verificar ausência de overlap (sem linhas em comum entre test e val)
test_rid_set = set(pd.read_parquet(TEST_SET, columns=['_row_id'])['_row_id'].tolist())
val_rid_set  = set(pd.read_parquet(VAL_SET,  columns=['_row_id'])['_row_id'].tolist())
overlap = test_rid_set & val_rid_set
print(f'\nOverlap test∩val: {len(overlap)} (deve ser 0)')

test_set:  1,654,191 linhas | anos: [2019, 2020, 2021, 2022, 2023, 2024]
val_set:   1,488,771 linhas  | anos: [2019, 2020, 2021, 2022, 2023, 2024]

Distribuição FAIXA_MT no teste:
faixa_3c
Baixo    46.8%
Medio    44.3%
Alto      9.0%
Name: proportion, dtype: object

Overlap test∩val: 0 (deve ser 0)
